# Train Llama 607M (Kazakh) from Scratch — A100 80GB

**Model**: Llama 607M (200K vocab, 24 layers, 1024 hidden, 16 heads)
**Data**: ~24.5B tokens (3 pre-tokenized datasets)
**GPU**: 1x A100 80GB
**Estimated time**: ~4-5 days with torch.compile

In [ ]:
!pip install -q transformers datasets accelerate huggingface_hub tokenizers

In [ ]:
# Login to HuggingFace
from huggingface_hub import login
login()  # paste your token

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

## Config

In [ ]:
# === CONFIG ===
TOKENIZER_NAME = "stukenov/sozkz-core-gpt2-200k-kk-base-v1"
DATASETS = [
    "stukenov/sozkz-corpus-tokenized-kk-200k-v1",
    "stukenov/sozkz-corpus-tokenized-kk-multidomain-200k-v1",
    "stukenov/sozkz-corpus-tokenized-enkk-200k-v1",
]
HF_PUSH_NAME = "stukenov/sozkz-llama-500m-kk-base-v1"
OUTPUT_DIR = "./outputs/exp018_llama_500m_200k"

# Training hyperparams (A100 80GB)
BATCH_SIZE = 16        # fits in 80GB
GRAD_ACCUM = 16        # effective batch = 256
LR = 3e-4
EPOCHS = 1
WARMUP_STEPS = 1000
EVAL_STEPS = 2000
SAVE_STEPS = 2000
USE_COMPILE = True     # ~1.5-2x speedup

## Load Pre-tokenized Datasets

In [ ]:
from datasets import Features, Sequence, Value, concatenate_datasets, load_dataset

target_features = Features({
    "input_ids": Sequence(Value("int32")),
    "labels": Sequence(Value("int32")),
})

train_parts, val_parts = [], []
for repo in DATASETS:
    print(f"Loading {repo}...")
    ds = load_dataset(repo)
    ds_train, ds_val = ds["train"], ds["validation"]
    print(f"  Train: {len(ds_train):,}, Val: {len(ds_val):,}")
    
    cols_to_keep = {"input_ids", "labels"}
    ds_train = ds_train.remove_columns([c for c in ds_train.column_names if c not in cols_to_keep])
    ds_val = ds_val.remove_columns([c for c in ds_val.column_names if c not in cols_to_keep])
    ds_train = ds_train.cast(target_features)
    ds_val = ds_val.cast(target_features)
    
    train_parts.append(ds_train)
    val_parts.append(ds_val)

train_dataset = concatenate_datasets(train_parts)
val_dataset = concatenate_datasets(val_parts)

print(f"\nCombined Train: {len(train_dataset):,}, Val: {len(val_dataset):,}")
print(f"Total tokens: ~{len(train_dataset) * 2048 / 1e9:.2f}B")

## Create Model

In [ ]:
from transformers import AutoTokenizer, LlamaConfig, LlamaForCausalLM

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config = LlamaConfig(
    vocab_size=200_019,
    hidden_size=1024,
    intermediate_size=4096,
    num_hidden_layers=24,
    num_attention_heads=16,
    num_key_value_heads=16,
    max_position_embeddings=2048,
    tie_word_embeddings=True,
    bos_token_id=2,
    eos_token_id=0,
    pad_token_id=1,
)
model = LlamaForCausalLM(config)

total_params = sum(p.numel() for p in model.parameters())
unique_params = sum(p.numel() for p in set(model.parameters()))
print(f"Model params: {total_params/1e6:.2f}M (unique: {unique_params/1e6:.2f}M)")

## Train

In [ ]:
import math
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

effective_batch = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = math.ceil(len(train_dataset) / effective_batch)
print(f"Effective batch: {effective_batch}")
print(f"Steps/epoch: {steps_per_epoch:,}")
print(f"Tokens/step: {effective_batch * 2048:,}")
print(f"Estimated total tokens: {steps_per_epoch * effective_batch * 2048 / 1e9:.2f}B")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=0.1,
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    bf16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    dataloader_num_workers=4,
    torch_compile=USE_COMPILE,
    report_to="none",
    push_to_hub=False,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [ ]:
import os, time

# Resume from checkpoint if exists
checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    checkpoints = [d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")]
    if checkpoints:
        latest = max(checkpoints, key=lambda x: int(x.split("-")[1]))
        checkpoint = os.path.join(OUTPUT_DIR, latest)
        print(f"Resuming from {checkpoint}")

t0 = time.time()
train_result = trainer.train(resume_from_checkpoint=checkpoint)
elapsed = time.time() - t0

print(f"\nTraining done in {elapsed:.0f}s ({elapsed/3600:.1f}h)")
print(f"Final loss: {train_result.metrics.get('train_loss', 0):.4f}")
print(f"Speed: {len(train_dataset) * 2048 / elapsed:.0f} tokens/sec")

## Evaluate & Save

In [ ]:
results = trainer.evaluate()
print(f"Eval loss: {results['eval_loss']:.4f}")
print(f"Perplexity: {2 ** results['eval_loss']:.2f}")

In [ ]:
# Save locally
final_dir = os.path.join(OUTPUT_DIR, "final")
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"Saved to {final_dir}")

In [ ]:
# Push to HuggingFace Hub
print(f"Pushing to {HF_PUSH_NAME}...")
model.push_to_hub(HF_PUSH_NAME)
tokenizer.push_to_hub(HF_PUSH_NAME)
print("Done!")

## Quick Generation Test

In [ ]:
from transformers import pipeline

gen = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)
prompts = [
    "Қазақстан — ",
    "Алматы қаласы ",
    "Білім алу ",
]
for p in prompts:
    out = gen(p, max_new_tokens=100, do_sample=True, temperature=0.8, top_p=0.9)
    print(f"\n{p}\n→ {out[0]['generated_text']}")